# Part 8 — The timeline, the literature, and what broke

The project ran from 2026-06-22 to 2026-08-12, organised as numbered **rounds**. A round
is one pre-registered question with its arms, seeds and selection rule fixed before
launch; it ends when its verification gate passes and its numbers enter the record.

Rounds themselves are fast — the deployable-scale training matrix of round 7 finished
thirty runs in seventy-five minutes of cluster time. This part is about everything else:
where the calendar time went, and the incidents that produced the rules in part 7.

![Rounds and incidents, 2026-06-22 to 2026-08-12. Green bars are training rounds on the cluster, gray bars local analysis campaigns, the purple bar the whole-model synthesis effort at the original scale; red markers are the incidents below. Generated by `tools/make_process_diagrams.py`.](figures/fig_rounds_timeline.png)

## The two eras

Everything divides at 2026-07-01, when training moved from a private two-class dataset to
the public HLS4ML LHC Jet five-class dataset. The move made every earlier accuracy number
incomparable with every later one, which is what the era firewall of part 7 enforces. The
rule costs a week of results on paper and prevents a table whose rows quietly disagree
about what they measure.

Era 1 still earned its keep. Its variant sweeps (rounds 1–4, 2026-06-27 to 06-29)
established that at that scale the optimizer recipe, not the architecture, drives tagging
performance — the round-1 architecture "wins" dissolved once the learning rate was tuned
per scale — and that verdict fixed the architecture for the rest of the project. The first
C-synthesis on mulder (2026-06-24) also predates the migration: the observation that
binary matmuls synthesize with zero DSPs was on record before the current dataset existed.

## The rounds of record (era 2)

| Round | Dates (2026) | The question | Outcome | Evidence |
|---|---|---|---|---|
| r5 | 07-01 – 07-03 | The quantization ladder on the fixed `large` architecture, baselines re-tuned at the same learning rate | First era-2 table; the rule "baselines may only get stronger" dates here | `bnjettag/roc-results/r5/` |
| HGQ2 rebuild | 07-04 – 07-06 | Can the trained model be reproduced in a QAT-native stack, bit-faithfully enough to trust? | Yes, after a custom binary quantizer; a stock 1-bit quantizer is ternary | `bnjettag/results/hgq2/` |
| Final campaign | 07-07 – 07-08 | The definitive matrix at the original scale: five variants, three seeds each, trained QAT-native | Fifteen runs trained and ROC-verified in two days, surviving two mid-campaign rescues | `bnjettag/roc-results/final/`, `bnjettag/results/final/` |
| Synthesis wall | 07-08 – 07-24 | Does the 6,375,173-parameter model synthesize as a whole? | No — four independent negatives; the wall is structural compiler capacity, not weights | `FULL-SYNTHESIS.md` |
| r7 / r7b | 07-13 – 07-16 | Retrain at deployable scale: 18,405- and 4,853-parameter models | Whole-model synthesis achieved; the tiny column retrained at its own tuned recipe | `bnjettag/r7/` |
| r8 | 07-13 – 07-19 | Was input scaling the real accuracy bug? A standardization-versus-normalization factorial | Yes: the norm-free, input-standardized 19,201-parameter flagship became the reference recipe | `bnjettag/r7/roc-results/r8/` |
| Compiler study | 07-20 – 07-26 | Is a binary-native adder-graph compiler the right next investment? | A measured negative on the naive version; the surviving levers quantified | `COMPILER.md`, `bnjettag/results/adder-graph/` |
| r10 | 07-24 – 07-25 | Do physics-informed pairwise features buy tagging performance? | Flat at six seeds; recorded as a negative and downgraded | `bnjettag/roc-results/r10/` |
| r11 | 07-25 – 07-26 | Does the binary model overtake 8-bit or FP32 as width grows (d32 to d128)? | It converges but never overtakes; a comparison confound was caught by the gate and fixed with control arms | `bnjettag/roc-results/r11/` |
| r13 | 08-01 | Learnable bitwidths under an explicit hardware-cost pressure | Learning rate re-pinned by a staged probe; the cost-accuracy front crossed the target | `bnjettag/code/hgq2/configs/` (r13) |
| r14 | 08-01 – 08-12 | The trigger-realistic input set: three features, constituent count swept over N ∈ {8, 16, 32, 64} | Sixty models verified; the whole-model zero-DSP point measured in synthesis at N = 8, RF = 1 | `bnjettag/roc-results/r14/`, `bnjettag/results/r14/hls_r14.md` |
| Replication | 08-04 – 08-12 | Reproduce the published grid of arXiv:2510.24784 from its released code | Eight of eight accuracy runs complete against a pinned revision, with paper-versus-code deltas documented | `bnjettag/results/repro-chang/` |

### Three numbers are missing, on purpose

- **r6** was designed (a normalization-cost study) but never launched; its question was
  absorbed into rounds 7 and 8, which answered it at the right scale.
- **r9** had its name consumed by a mid-July learning-rate sweep, so the pairwise-bias
  experiment became round 10.
- **r12**, a knowledge-distillation arm with a pre-registered promote/kill rule, was
  implemented (`bnjettag/code/hgq2/distill_r12.py`) but never run: its launch checklist was
  not completed, and launching an unverified job would have violated the preflight rule it
  depended on.

An honest ledger records what was designed and not run, because an unnumbered gap invites
the suspicion that something was run and not reported.

## Where the time went

Training was never the bottleneck. Three things were.

**1. The synthesis wall — about two weeks, mid-July.** Four separate strategies for
synthesizing the 6.4-million-parameter model each consumed days of synthesis time before
returning a negative (`FULL-SYNTHESIS.md`). The wall ended only when the question was
inverted: on 2026-07-13 the model itself, not the toolchain, was declared the bug, and the
project retrained at a scale with published precedent. In retrospect those two weeks
bought the single most consequential insight of the project; at the time it looked like
failure.

**2. Infrastructure, twice.** The cluster-wide storage outage of 2026-07-02 forced a
redesign of every job to run without persistent volumes, and the silent loss of four
checkpoints on 2026-07-15 forced the artifact-upload contract. Both redesigns outlived
their incidents — the PVC-free, artifact-first job pattern is simply how every run works
now — but each cost real days when it happened.

**3. Verification, on purpose.** Every accuracy number was recomputed from stored score
arrays before being quoted, every hardware number re-parsed from the raw synthesis report,
and no gap between variants was reported without seed spread and a paired bootstrap. This
overhead took a deliberate and substantial share of all effort, and it repeatedly changed
the record: the round-11 ladder passed through three successive gate reviews that produced
four corrections and one withdrawn claim before its table stabilized, and the round-14
folded-synthesis table went through three passes in a single day, 2026-08-12 — fail,
still-draft, verified — before it was allowed into
`bnjettag/results/r14/hls_r14_folded.md`.

Time spent this way is why the tables that survive can be defended line by line.

![Accuracy against measured hardware cost at the end of the timeline: the round-14 Pareto front over the constituent-count sweep. Generated by `bnjettag/code/plots/make_r14_plots.py` from `bnjettag/roc-results/r14/` and `bnjettag/results/r14/ebops_r14.json`.](figures/fig_r14_pareto.png)

## Working the literature

Literature here is worked, not collected. A paper enters the record only when it has been
read to a stated depth and its relevance written down.

- **Briefs — one page per paper.** The library under `docs/literature/` holds 53 brief
  files covering 50 distinct papers, in six topic folders: the BitNet line and its ternary
  ancestors (6), jet-tagging transformers (15), hls4ml and FPGA triggers (8), QAT and
  binary-network foundations (8), compiler/IR/hardware (6), and the LUT-native family
  (10). Each brief ends with a "Relevance to BNJetTag" section — what the paper is *to
  this project*, not what it is in general. A brief is the minimum ticket of entry: no
  paper is cited anywhere in the record without one.
- **Deep reads.** When a paper bears on a live decision it gets a dossier: what they did,
  the numbers that matter here, what transfers, what threatens the thesis. The closest
  published system, the sub-microsecond FPGA transformers of arXiv:2510.24784, carries
  two documents — the science dossier and a companion written to state exactly what can
  and cannot be replicated. A deep read is triggered by stakes, not by fame.

### Reading that changed what got built

**The deployable-scale pivot.** Through early July the flagship was a 6.4M-parameter
model, and the synthesis evidence memo of 2026-07-09 had already ranked "small-model full
synthesis" as the leading option, observing that the sub-microsecond paper's models
synthesize whole because they are sized for it
(`docs/reports/evidence-hls4ml-size-strategy-2026-07-09.md`). The published reference
points said the same thing from the accuracy side: constituent-based taggers on this
dataset cluster in the low tens of thousands of parameters, JEDI-net at ~34k, and the
sub-µs transformers trained to a 350k-EBOPs budget against this project's 530.4M
(`bnjettag/results/ebops.md` §3). On 2026-07-13 that survey triggered the decision — the
6.4M model was retired as not an L1 trigger model, and rounds 7 and 8 retrained the thesis
at roughly 19k and 5k parameters. Within two days the whole small model had synthesized as
a single Vitis project (`RESEARCH.md` §2). Reading did not decorate that decision; it
caused it.

**Replication as reading.** Sometimes the only way to finish reading a paper is to run its
code. The replication of arXiv:2510.24784 targets the released repository at a pinned
commit (`6cdc6e3`), because the repository, eight months downstream of the paper, deviates
from the paper's own description in load-bearing ways. Three deltas travel with every
quoted number (`bnjettag/results/repro-chang/comparison.md`): the released transformer
config uses two attention heads where the paper says single-head; the released `xfmt`
variant is a post-paper LUT-based Linformer, not the Linformer the paper describes; and
the released training uses an open-loop piecewise β schedule where the paper claims a PID
controller. The honest replication target is the repo at the pinned SHA with the delta
list attached.

## The lessons

Every rule in part 7, and most of the contracts in the earlier parts, exists because
something specific broke.
Two kinds of entry appear below. **Incidents** failed and were repaired. **Negatives** ran
correctly and returned "no" — those are results, kept to the same evidentiary standard as
any positive, because a negative someone can trust is worth more than a positive nobody
checked.

Each entry reads the same way: symptom, cause, fix, and the rule it left behind.

**The 6.4M model would not synthesize — attempts one and two (2026-07-08, 07-11)**

- **Symptom:** a single-project synthesis folded at RF = 256 survived the compiler front
  end at roughly 27 million intermediate-representation instructions, then died silently
  in the Unroll/Inline phase — process gone, logs frozen mid-line. Rerouting the weights
  into block RAM ran 32 hours with memory flat at 54 GB and died in the same phase at
  27,941,780 front-end instructions.
- **Cause:** the near-equality of the two instruction counts. The count is
  structure-dominated, not weight-dominated, so no emission trick that reorganizes weights
  can pass the wall (`FULL-SYNTHESIS.md`).
- **Fix:** none at this scale; the third and fourth routes below, then the pivot.
- **Rule it left behind:** measure the failure mode before designing the next attempt — the
  first run's silent death looked like an out-of-memory kill on a 125 GB machine, and only
  the second run's identical count identified what the limit actually was.

**The hierarchical route livelocked (killed 2026-07-24)**

- **Symptom:** per-block hierarchical synthesis entered the problematic phase at a modest
  3.5 million instructions and never converged. Killed after 13.9 days of wall clock, nine
  of which had produced no persisted output at all.
- **Cause:** a livelock, not slow progress (`FULL-SYNTHESIS.md`, Route B verdict).
- **Fix:** the question was inverted instead. On 2026-07-13 the model was declared the
  bug — six million parameters and 63.4 million multiply-accumulates per jet
  (`bnjettag/results/ebops.md` §1) is not a Level-1 trigger model under any toolchain — and
  the project retrained at 18,405 and 4,853 parameters (`bnjettag/r7/README.md`).
- **Rule it left behind:** a long job with no persisted output is not progress; and every
  whole-model synthesis claim in the record now lives exclusively at deployable scale.

**Distributed arithmetic made the binary model bigger (negative)**

- **Symptom:** the constant-sharing strategy that pays for itself on multi-bit fixed-point
  weights *inflated* the whole model to 4.53 million LUTs (+16%) at 3.7 times the
  flip-flops (`RESEARCH.md` §2, RF = 8 comparison).
- **Cause:** the canonical signed-digit form of a ±1 weight has exactly one nonzero digit,
  so the sharing opportunities that pay for that strategy's pipelining discipline do not
  exist at the binary extreme, while its costs remain.
- **Fix:** none needed — the naive inlined adder tree was already optimal. The census is
  in `bnjettag/results/adder-graph/`.
- **Rule it left behind:** a lever that works for 8-bit weights must be re-measured at
  1 bit, not assumed; this negative reshaped the compiler design study (`COMPILER.md`).

**The quantizer that was secretly ternary (2026-07-04)**

- **Symptom:** a probe of the stock 1-bit fixed-point quantizer on real checkpoint
  statistics collapsed 4,096 of 4,096 small latent weights to exactly zero
  (`CONTRIBUTIONS.md` §1.1).
- **Cause:** a 1-bit *fixed-point* grid is {−1, 0, +1}. Bipolar sets are not fixed-point
  sets.
- **Fix:** the custom absmean quantizer, with a strict bipolar sign (zero maps to +1, never
  to 0) and an export-time counter asserting that the number of sign-zeros is exactly zero.
- **Rule it left behind:** "silently off-thesis" is a class of failure the pipeline tests
  for explicitly. A ternary model would have voided the thesis while every downstream tool
  reported success.

**The A4 arm, failure one: NaN at the first step (2026-07-07)**

- **Symptom:** the 4-bit-activation arm produced a NaN at the first training step, at every
  learning rate tried. Halving the rate twice changed nothing, which ruled out the usual
  diagnosis.
- **Cause:** the gradient-clipping mode. Global-norm clipping must first *square* the
  global gradient norm; the normalization layers' backward Jacobian on adversarial 4-bit
  batches reaches magnitudes around $2.5 \times 10^{20}$, and squaring that overflows
  32-bit floating point. The clip meant to protect the step failed exactly when it was
  needed.
- **Fix:** element-wise value clipping, which involves no global reduction and cannot
  overflow, applied uniformly across all five variants so the comparison matrix stayed on
  one clipping rule. A related hardening routed the straight-through gradient through a
  stop-gradient on the β scale, removing a 1/β amplification that blows up when a weight
  tensor's spread collapses.
- **Rule it left behind:** a safety mechanism containing a reduction can fail in exactly
  the regime it exists for; and a fix to one arm is applied to every arm of the matrix.

**The A4 arm, failure two: frozen activation grids (2026-07-07)**

- **Symptom:** with training stable, the A4 arm converged to a validation macro-OvR AUC of
  0.59 (per-seed 0.5825 and 0.6049). Collapse, not noise.
- **Cause:** the activation grids were calibrated once on the activations of an untrained
  network and then frozen. As training moved the activation distributions, the frozen grids
  went misaligned. At 8 bits (256 levels) that is absorbed; at 4 bits (16 levels) it is
  fatal.
- **Fix:** make the grid scales trainable during QAT while keeping the exported form
  identical. This recovered the full 16.8 validation points (`RESEARCH.md` §5″); the
  frozen-grid runs were kept, relabelled, as the ablation evidence.
- **Rule it left behind:** a calibration performed at initialization is a hypothesis about
  the trained network, and at aggressive quantization it is a wrong one.

**The storage outage and the PVC-free pivot (2026-07-02)**

- **Symptom:** the cluster's shared filesystem backend failed cluster-wide and the
  evaluation job blocked on a volume that would never mount — 39 aborted mount attempts
  over 70 minutes.
- **Cause:** the platform's, not the project's. But waiting was not a plan.
- **Fix:** rebuild the job to need nothing from the broken volume: the trained checkpoints
  already existed in W&B and the evaluation data is public on Zenodo. Five days later, when
  training jobs mysteriously would not schedule on GPU nodes, an A/B probe — two pods,
  identical except for the volume claim — showed the claim itself was filtering pods off
  GPU nodes, the outage's half-healed tail, and the whole campaign moved to the same
  volume-free pattern.
- **Rule it left behind:** data fetched in-pod with a size gate, code shipped by ConfigMap,
  outputs leaving exclusively through versioned artifacts. The emergency workaround proved
  strictly better than the original design and became the standing architecture.

**The checkpoints that vanished (2026-07-15)**

- **Symptom:** an evaluation job failed its own assertion that exactly one checkpoint file
  exists per run.
- **Cause:** four of eight recently finished training runs had ended green, logged "files
  synced", and left a zero-byte checkpoint in W&B — an end-of-run upload race on pods whose
  local disks had already evaporated. Four trained models were unrecoverable and had to be
  retrained.
- **Fix:** a durability gate in the trainer — after the run closes, re-query the uploaded
  file by size through the public API, re-upload on mismatch, exit nonzero after five
  failed attempts. On 2026-08-01 this hand-rolled gate was replaced by the artifact system
  proper, where uploads block on a server-side commit and the manifest is asserted.
- **Rule it left behind:** a green job no longer means "the pod finished", only "the
  artifact is confirmed durable". The versioned artifact, not the pod, is the checkpoint of
  record.

**One bad node, and a live ConfigMap (2026-07-25)**

- **Symptom:** 17 of 27 newly launched round-11 jobs failed within twenty minutes, across
  all arms and scales, for configurations that had trained fine as probes. Eleven more died
  at startup later the same day.
- **Cause:** every failed pod in the first group carried the same device-plugin admission
  error and every one had landed on the *same* node — one broken machine absorbing pod
  after pod, and with a retry budget of two, three admission bounces retire a job before it
  runs an instruction. The second group followed a code ConfigMap refreshed three times in
  quick succession while jobs were pending: ConfigMap updates propagate to mounted volumes
  with a sync delay, so pods starting inside that window unpacked the previous code tree
  and failed the stale-code gate three times.
- **Fix:** the node went onto an explicit blocklist in the job generator, and the affected
  jobs were deleted and recreated rather than patched, because a Job's pod template is
  immutable.
- **Rule it left behind:** build the ConfigMap once, before launching; if a round needs new
  code mid-flight, give it a new name and point only new jobs at it. The stale-code gate
  itself behaved as designed — stale code died loudly instead of running silently.

**GATE 1 at 0.75, and an honest ceiling at 0.989 (2026-07-18)**

- **Symptom:** the first norm-free export scored a GATE-1 score correlation of 0.75 against
  the trained model. Catastrophic.
- **Cause:** activation grids sized for the wrong stream statistics.
- **Fix:** the repaired export reached 0.989, and the remaining gap to the nominal 0.9999
  bar was characterized rather than waved through: the limiter is the signed-digit encoding
  of β products on a graph with no normalization layer to absorb the rounding, and even an
  exact-β reproduction ceilings at 0.994 on this graph.
- **Rule it left behind:** characterize the gap, never relax the threshold. The record
  carries 0.989 as a documented ceiling with `gate1_pass: false` stored in the artifact
  (`bnjettag/r7/results/convert/r8stdnn-s2/rf1/export_verify.json`, quoted in
  `RESEARCH.md` §2).

**The round-14 export that widened the wrong grids (2026-08-04)**

- **Symptom:** the round-14 exports failed GATE 1 outright.
- **Cause:** a layer-by-layer correlation trace localized the cliff to the attention output
  projection and found a cause nobody had hypothesized — the export pipeline's
  grid-widening step, added in July precisely to fix the previous failure, was removing
  saturation the trained model had learned to rely on. The clipping is part of the learned
  function; an export that "generously" widens those grids reproduces the arithmetic and
  destroys the nonlinearity.
- **Fix:** no small code change was correct. β had to return to the graph as explicit
  affine layers. The rebuilt export ("v5") passed at 0.9988 on the 8-constituent article
  and 0.9974 on the 16-constituent one, with the encoding chosen by a selection rule
  pre-registered before the ladder was measured.
- **Rule it left behind:** a bug report is a claim like any other. Two days earlier this
  diagnosis had been independently re-verified from fresh runs before any fix was
  committed.

**The 256-DSP accumulator trap (pinned 2026-07-09)**

- **Symptom:** under hls4ml's Resource strategy, with weights served from block RAM instead
  of inlined as constants, a 256×256 "binary" matmul synthesized to 256 real DSPs at
  RF = 256 — one multiplier per output row, on a layer the thesis says must use none.
- **Cause:** not the weights. With pure ±1 weights in ROM the DSPs are set by the *dense
  accumulator*, whose narrow width binds to a DSP48 where a mathematically-exact widened
  one maps to LUTs.
- **Fix:** structurally, keep the datapath weights pure ±1 and move the scale and bias into
  a frozen affine, confirmed at zero DSPs; then the enabling conditions were written down
  as a reusable support matrix rather than left as folklore (`CONTRIBUTIONS.md` §2.4,
  `bnjettag/results/hgq2/constraints_map.md`).
- **Rule it left behind:** the per-module census — attribute every DSP to a named module,
  never trust the total. Only the per-instance report caught this.

**The parity gate caught an emission bug (2026-07-24)**

- **Symptom:** in the compiler study, a hand-emitted parity control came back 80% over its
  predicted LUT count on first launch.
- **Cause:** the emitted code seeded every summation chain from the wide accumulator type,
  forcing every addition in the balanced tree to full width instead of growing from 8-bit
  leaves. Measured cost of that one idiom: about 1.8× in LUTs
  (`bnjettag/results/adder-graph/README.md`).
- **Fix:** grow the tree from its leaves; the post-fix parity arm reads 1.151× against the
  census (`bnjettag/results/adder-graph/e1/RESULTS.md`).
- **Rule it left behind:** the control arm must reproduce the known census before any
  experimental arm counts. That gate is the only reason a subtle emission bug did not
  contaminate every downstream comparison of the campaign.

**Smaller negatives worth their lines**

Some entries earn one sentence precisely because they were caught early.

- The upstream reference model's "learned" positional encoding was found (2026-06-27) to
  train zero parameters — a silent no-op inherited by anyone who copies the code without
  counting.
- The round-10 pairwise-feature bias, physically motivated, was flat at six seeds and is
  recorded as a negative rather than tuned until it confessed (`bnjettag/roc-results/r10/`).
- A fold-composition probe wedged for thirteen days before being declared dead on
  2026-08-08 with its question explicitly unanswered rather than guessed.
- The first launch of the replication campaign was generated with a constituent count read
  off a dataset file name (the dataset ships in its 150-particle form). Caught in review
  before launch, regenerated, and kept in the log as a reminder that file names are not
  metadata.
- The synthesis driver on mulder was discovered on 2026-08-04 to have been silently
  ignoring the stage flags passed on its command line — every "synthesis-only" run had also
  been running hours of unrequested simulation stages. Fixed by writing the stage table
  into the file the build script actually sources (`STATUS-2026-08-08.md` §3.5).

## What is still broken

The honest state of the open problems, frozen as of 2026-08-08, is section 3 of
`STATUS-2026-08-08.md`:

1. The 16-constituent whole-model synthesis exceeds what the synthesis machine can
   complete.
2. Device fit at RF = 1 is unsolved: the maximum-parallelism points exceed the target
   device's LUT budget.
3. The surviving DSPs live in the β affines.
4. The selection margin on the 16-constituent article is thin.
5. Two EBOPs conventions coexist and must never share a column.
6. The replication campaign's numbers had not yet passed the verification gate at that
   snapshot.
7. A set of July-built tooling was never completed to the launch bar.

A problem inventory with named, dated, unresolved entries is what a real research record
looks like from inside. The alternative — a document that resolves everything — would mean
the gates had stopped finding anything, and that would be the loudest warning of all.